### Cell 1 — Install dependencies

In [ ]:
!pip -q install rdflib pillow "pydantic>=2,<3" python-slugify tqdm requests


### Cell 2 — Config (edit your paths here)

In [ ]:
from pathlib import Path

# === Input paths (edit BASE_INPUT to your dataset) ===
BASE_INPUT = Path("/kaggle/input/your-dataset-name")  # <-- EDIT THIS
FEWSHOT_JSON = BASE_INPUT / "rdf_extractions_FewShot.json"
ONESHOT_JSON = BASE_INPUT / "rdf_extractions_oneShot.json"
IMAGES_DIR   = BASE_INPUT / "images"   # folder containing images

# === Output path ===
OUT_DIR = Path("/kaggle/working/rdf_eval_out")
OUT_DIR.mkdir(parents=True, exist_ok=True)

# === AvalAI config ===
AVALAI_BASE_URL = "https://api.avalai.ir/v1"
AVALAI_SECRET_NAME = "avalai_api"  # Kaggle secret key name

# === Models ===
# Judge-1 (Gemma) for Pointwise + Pairwise
GEMMA_MODEL = "gemma-3n-e4b-it"
# Meta-Judge (Qwen) for the final reconciliation
QWEN_MODEL  = "qwen3-235b-a22b"  # adjust to your exact AvalAI model name if different

print("Few-shot JSON:", FEWSHOT_JSON.exists(), FEWSHOT_JSON)
print("One-shot JSON:", ONESHOT_JSON.exists(), ONESHOT_JSON)
print("Images ZIP:", IMAGES_DIR.exists(), IMAGES_DIR)
print("Output dir:", OUT_DIR)
print("AvalAI base URL:", AVALAI_BASE_URL)
print("Gemma model:", GEMMA_MODEL)
print("Qwen model:", QWEN_MODEL)


### Cell 3 — Imports & seed

In [ ]:
import os, io, re, json, zipfile, base64, random, csv, glob
from dataclasses import dataclass
from typing import Any, Dict, List, Optional, Tuple

from pydantic import BaseModel, field_validator, ValidationError
from rdflib import Graph
from PIL import Image
from slugify import slugify
from tqdm import tqdm

random.seed(7)


### Cell 4 — Stage-0 deterministic checks (syntax, hygiene)

In [ ]:
@dataclass
class Stage0Signals:
    parses: bool
    triple_count: int
    namespaces: List[str]
    parse_error: Optional[str]
    has_blank_nodes: bool
    duplicate_triples_estimate: int
    unknown_prefixes_count: int

def run_stage0_checks(turtle_text: str) -> Stage0Signals:
    g = Graph()
    try:
        g.parse(data=turtle_text, format="turtle")
        parses = True
        err = None
    except Exception as e:
        parses = False
        err = str(e)

    triple_count = len(g) if parses else 0
    namespaces = [str(pfx) for (pfx, _) in (g.namespaces() if parses else [])]

    n_blank = 0
    dup_est = 0
    if parses:
        seen = set()
        for s, p, o in g:
            if str(s).startswith("_:") or str(o).startswith("_:"):
                n_blank += 1
            key = (str(s), str(p), str(o))
            if key in seen:
                dup_est += 1
            else:
                seen.add(key)

    token_prefixes = set(re.findall(r'([A-Za-z][A-Za-z0-9_\-]*)\:', turtle_text))
    declared = set(p.split(":")[0] for p in namespaces if ":" in p)
    unknown = 0
    for tp in token_prefixes:
        if tp in {"http", "https"}:
            continue
        if tp not in declared:
            unknown += 1

    return Stage0Signals(
        parses=parses,
        triple_count=triple_count,
        namespaces=sorted(namespaces),
        parse_error=err,
        has_blank_nodes=(n_blank > 0),
        duplicate_triples_estimate=dup_est,
        unknown_prefixes_count=unknown
    )

def summarize_signals(signals: Stage0Signals) -> str:
    return json.dumps({
        "parses": signals.parses,
        "triple_count": signals.triple_count,
        "namespaces": signals.namespaces,
        "parse_error": signals.parse_error or "",
        "has_blank_nodes": signals.has_blank_nodes,
        "duplicate_triples_estimate": signals.duplicate_triples_estimate,
        "unknown_prefixes_count": signals.unknown_prefixes_count
    }, ensure_ascii=False)


### Cell 5 — Image helpers (read from ZIP, base64 JPEG)

In [ ]:
def load_image_bytes(folder_path: Path, image_name: str) -> Optional[bytes]:
    if not folder_path or not folder_path.exists():
        return None
    # Try exact file
    candidate = folder_path / image_name
    if candidate.exists():
        return candidate.read_bytes()
    # Try case-insensitive / suffix match
    lower = image_name.lower()
    for f in folder_path.glob("**/*"):
        if f.name.lower() == lower or f.name.lower().endswith(lower):
            return f.read_bytes()
    return None


def image_to_base64_jpeg(image_bytes: bytes, max_side: int = 1536) -> str:
    img = Image.open(io.BytesIO(image_bytes)).convert("RGB")
    w, h = img.size
    scale = min(max_side / max(w, h), 1.0)
    if scale < 1.0:
        img = img.resize((int(w*scale), int(h*scale)))
    out = io.BytesIO()
    img.save(out, format="JPEG", quality=90)
    return base64.b64encode(out.getvalue()).decode("utf-8")


### Cell 6 — Pydantic schemas for scores & meta

In [ ]:
class PointwiseScore(BaseModel):
    accuracy: int
    entity_linking: int
    relation_correctness: int
    completeness: int
    consistency_validity: int
    clarity: int
    final_score: int
    rationale: str

    @field_validator(
        "accuracy","entity_linking","relation_correctness",
        "completeness","consistency_validity","clarity","final_score"
    )
    @classmethod
    def _0to10(cls, v: int) -> int:
        v = int(v)
        if not (0 <= v <= 10):
            raise ValueError("Scores must be integers in [0,10].")
        return v

class PairwiseDecision(BaseModel):
    preferred_response: str  # "A" | "B" | "tie"
    preference_strength: str # "slight" | "moderate" | "strong" | "none"
    rationale: str

    @field_validator("preferred_response")
    @classmethod
    def pref_valid(cls, v: str) -> str:
        if v not in {"A","B","tie"}:
            raise ValueError("preferred_response must be 'A' | 'B' | 'tie'")
        return v

    @field_validator("preference_strength")
    @classmethod
    def strength_valid(cls, v: str) -> str:
        if v not in {"slight","moderate","strong","none"}:
            raise ValueError("preference_strength must be slight|moderate|strong|none")
        return v

class FinalPointwisePerResponse(BaseModel):
    accuracy: int
    entity_linking: int
    relation_correctness: int
    completeness: int
    consistency_validity: int
    clarity: int
    final_score: int

class MetaFinal(BaseModel):
    final_pointwise: Dict[str, FinalPointwisePerResponse]  # keys "A","B"
    final_pairwise: PairwiseDecision
    meta_rationale: str


### Cell 7 — Read JSONs & compute overlap

In [ ]:
def read_json_list(path: Path) -> List[Dict[str, Any]]:
    with open(path, "r", encoding="utf-8") as f:
        data = json.load(f)
    if isinstance(data, dict) and "dataset" in data:
        return data["dataset"]
    if isinstance(data, list):
        return data
    raise ValueError(f"Unrecognized JSON structure: {path}")

few_list = read_json_list(FEWSHOT_JSON)
one_list = read_json_list(ONESHOT_JSON)

map_few = {x["source_image"]: x["rdf_graph_turtle"] for x in few_list}
map_one = {x["source_image"]: x["rdf_graph_turtle"] for x in one_list}

overlap = sorted(set(map_few).intersection(map_one))
print(f"Few-shot entries: {len(few_list)}")
print(f"One-shot entries: {len(one_list)}")
print(f"Overlapping images: {len(overlap)}")


### Cell 8 — LLM adapters (Mock + stubs for real Gemma/Gemini)

In [ ]:
import requests
from kaggle_secrets import UserSecretsClient

class LLMClient:
    def generate(self, prompt: str, *, image_b64_jpeg: Optional[str] = None, response_format_json: bool = True) -> str:
        raise NotImplementedError

class AvalAIAdapter(LLMClient):
    """
    Adapter for AvalAI API (assumed OpenAI-compatible /chat/completions endpoint).

    Requires:
      - Kaggle Secret "avalai_api"
      - BASE_URL like https://api.avalai.ir/v1
      - model_name e.g., "gemma-3n-e4b-it" or "qwen-72b-instruct"
    """
    def __init__(self, model_name: str, temperature: float = 0.0,
                 base_url: str = "https://api.avalai.ir/v1", timeout: int = 60,
                 secret_name: str = "avalai_api"):
        self.model_name = model_name
        self.temperature = temperature
        self.base_url = base_url.rstrip("/")
        self.timeout = timeout

        # Retrieve API key securely from Kaggle Secrets
        user_secrets = UserSecretsClient()
        self.api_key = user_secrets.get_secret(secret_name)
        if not self.api_key:
            raise RuntimeError(f"Kaggle Secret '{secret_name}' not found. Please add it in Add-ons → Secrets.")

        self.endpoint = f"{self.base_url}/chat/completions"
        self.headers = {
            "Authorization": f"Bearer {self.api_key}",
            "Content-Type": "application/json",
        }

    def generate(self, prompt: str, *, image_b64_jpeg: Optional[str] = None, response_format_json: bool = True) -> str:
        """
        Sends a single-turn user message to AvalAI.
        """
        payload = {
            "model": self.model_name,
            "messages": [
                {"role": "user", "content": prompt}
            ],
            "temperature": self.temperature,
            "max_tokens": 1024
        }

        resp = requests.post(self.endpoint, headers=self.headers, json=payload, timeout=self.timeout)
        if resp.status_code != 200:
            raise RuntimeError(f"AvalAI request failed ({resp.status_code}): {resp.text[:500]}")

        data = resp.json()
        try:
            return data["choices"][0]["message"]["content"]
        except Exception:
            return json.dumps(data)

# Optional mock adapter (offline testing only)
class MockAdapter(LLMClient):
    def __init__(self, seed: int = 7):
        random.seed(seed)
    def generate(self, prompt: str, *, image_b64_jpeg: Optional[str] = None, response_format_json: bool = True) -> str:
        if "preferred_response" in prompt or "Compare two RDF" in prompt or "Compare two RDF graphs" in prompt:
            choice = "A" if prompt.lower().count("turtle") % 2 == 0 else "B"
            strength = random.choice(["slight","moderate","strong"])
            return json.dumps({"preferred_response": choice, "preference_strength": strength, "rationale": "Mock pairwise."})
        if "Meta-Judge" in prompt or '"final_pointwise"' in prompt:
            return json.dumps({
                "final_pointwise": {
                    "A": {"accuracy": 8,"entity_linking": 7,"relation_correctness": 8,"completeness": 7,"consistency_validity": 8,"clarity": 7,"final_score": 8},
                    "B": {"accuracy": 6,"entity_linking": 6,"relation_correctness": 6,"completeness": 6,"consistency_validity": 7,"clarity": 6,"final_score": 6}
                },
                "final_pairwise": {"preferred_response": "A","preference_strength": "moderate","rationale":"Mock"},
                "meta_rationale": "Mock meta-consistency."
            })
        return json.dumps({
            "accuracy": 7,"entity_linking": 7,"relation_correctness": 7,
            "completeness": 7,"consistency_validity": 7,"clarity": 7,
            "final_score": 7,"rationale": "Mock pointwise."
        })


### Cell 9 — Prompt templates + JSON extractor

In [ ]:
POINTWISE_PROMPT = """You are Judge-1 (Gemma). Evaluate a single RDF graph extracted from an image.
Return STRICT JSON ONLY with the keys below.

Criteria (0-10 each):
- accuracy: factual correctness of triples against the image content
- entity_linking: correct entity labels/URIs; class usage consistent
- relation_correctness: predicates and directionality are appropriate
- completeness: important facts from the image are captured
- consistency_validity: RDF parses; namespaces sensible; no contradictions
- clarity: label quality, minimal redundancy, graph readability
- final_score: overall score (holistic)

JSON schema:
{
  "accuracy": 0-10,
  "entity_linking": 0-10,
  "relation_correctness": 0-10,
  "completeness": 0-10,
  "consistency_validity": 0-10,
  "clarity": 0-10,
  "final_score": 0-10,
  "rationale": "short explanation"
}

[Task]: Extract image facts as RDF (Turtle).
[Stage-0 signals]: {signals}
[Image note]: {image_note}
[RDF_Turtle]:
```
{turtle}

```

"""

In [ ]:
PAIRWISE_PROMPT = """You are Judge-1 (Gemma). Compare two RDF graphs (A vs B) extracted from the SAME image.
Return STRICT JSON ONLY:

{
  "preferred_response": "A" | "B" | "tie",
  "preference_strength": "slight" | "moderate" | "strong" | "none",
  "rationale": "short explanation"
}

Criteria for comparison:
1) Accuracy (triples match the image content)
2) Coherence (graph structure, ontology fit, organization)
3) Style/Clarity (labels, conciseness, readability)

[Stage-0 signals A]: {signals_a}
[Stage-0 signals B]: {signals_b}
[Image note]: {image_note}

[Response A - Turtle]
```
{turtle_a}
```

[Response B - Turtle]
```
{turtle_b}
```
"""

In [ ]:
META_PROMPT = """You are the Meta-Judge (Qwen). You receive:
- Judge-1 pointwise for A and B
- Judge-1 pairwise
- Same image task and responses

Resolve inconsistencies if any and return STRICT JSON ONLY:

{
  "final_pointwise": {
    "A": {"accuracy": 0-10, "entity_linking": 0-10, "relation_correctness": 0-10, "completeness": 0-10, "consistency_validity": 0-10, "clarity": 0-10, "final_score": 0-10},
    "B": {"accuracy": 0-10, "entity_linking": 0-10, "relation_correctness": 0-10, "completeness": 0-10, "consistency_validity": 0-10, "clarity": 0-10, "final_score": 0-10}
  },
  "final_pairwise": {
    "preferred_response": "A" | "B" | "tie",
    "preference_strength": "slight" | "moderate" | "strong" | "none"
  },
  "meta_rationale": "why the decision is coherent"
}

[Pointwise A]: {pointwise_a}
[Pointwise B]: {pointwise_b}
[Pairwise]: {pairwise}
[Image note]: {image_note}


Response A - Turtle]

```
{turtle_a}
```

[Response B - Turtle]
```
{turtle_b}
```
"""

In [ ]:

def ensure_json_object(text: str) -> Dict[str, Any]:
    # Extract first JSON object; supports optional fenced code blocks
    m = re.search(r"```(?:json)?\s*(\{.*?\})\s*```", text, flags=re.S)
    if m:
        text = m.group(1)
    m2 = re.search(r"\{.*\}", text, flags=re.S)
    if not m2:
        raise ValueError("No JSON object found in model output.")
    return json.loads(m2.group(0))


### Cell 10 — Core pipeline for one image (Stage-0 → Stage-1 → Stage-2)

In [ ]:
def run_pipeline_for_image(
    image_name: str,
    turtle_a: str, turtle_b: str,
    image_b64_jpeg: Optional[str],
    gemma: LLMClient,   # Judge-1 (Pointwise + Pairwise)
    qwen: LLMClient,    # Meta-Judge
    out_dir: Path
) -> Dict[str, Any]:
    # Stage-0
    s0_a = run_stage0_checks(turtle_a)
    s0_b = run_stage0_checks(turtle_b)
    image_note = "Image provided (base64 JPEG)" if image_b64_jpeg else "Image not available or not used."

    # Stage-1 Pointwise (A)
    p_prompt_a = POINTWISE_PROMPT.format(
        signals=summarize_signals(s0_a),
        image_note=image_note,
        turtle=turtle_a
    )
    p_raw_a = gemma.generate(p_prompt_a, image_b64_jpeg=image_b64_jpeg, response_format_json=True)
    p_json_a = ensure_json_object(p_raw_a)
    pscore_a = PointwiseScore.model_validate(p_json_a)

    # Stage-1 Pointwise (B)
    p_prompt_b = POINTWISE_PROMPT.format(
        signals=summarize_signals(s0_b),
        image_note=image_note,
        turtle=turtle_b
    )
    p_raw_b = gemma.generate(p_prompt_b, image_b64_jpeg=image_b64_jpeg, response_format_json=True)
    p_json_b = ensure_json_object(p_raw_b)
    pscore_b = PointwiseScore.model_validate(p_json_b)

    # Stage-1 Pairwise
    pair_prompt = PAIRWISE_PROMPT.format(
        signals_a=summarize_signals(s0_a),
        signals_b=summarize_signals(s0_b),
        image_note=image_note,
        turtle_a=turtle_a,
        turtle_b=turtle_b
    )
    pair_raw = gemma.generate(pair_prompt, image_b64_jpeg=image_b64_jpeg, response_format_json=True)
    pair_json = ensure_json_object(pair_raw)
    pair = PairwiseDecision.model_validate(pair_json)

    # Stage-2 Meta-Judge (Qwen)
    meta_prompt = META_PROMPT.format(
        pointwise_a=json.dumps(pscore_a.model_dump()),
        pointwise_b=json.dumps(pscore_b.model_dump()),
        pairwise=json.dumps(pair.model_dump()),
        image_note=image_note,
        turtle_a=turtle_a,
        turtle_b=turtle_b
    )
    meta_raw = qwen.generate(meta_prompt, image_b64_jpeg=image_b64_jpeg, response_format_json=True)
    meta_json = ensure_json_object(meta_raw)
    meta = MetaFinal.model_validate(meta_json)

    # Save per-image result
    base = slugify(Path(image_name).name)
    out_path = out_dir / f"{base}.final.json"
    with open(out_path, "w", encoding="utf-8") as f:
        json.dump({
            "image": image_name,
            "stage0": {"A": s0_a.__dict__, "B": s0_b.__dict__},
            "pointwise_raw": {"A": p_json_a, "B": p_json_b},
            "pairwise_raw": pair.model_dump(),
            "final": meta.model_dump()
        }, f, ensure_ascii=False, indent=2)

    return {
        "image": image_name,
        "final": meta.model_dump(),
        "pairwise": pair.model_dump(),
        "pointwise_A": pscore_a.model_dump(),
        "pointwise_B": pscore_b.model_dump()
    }


### Cell 11 — Aggregation & reporting helpers

In [ ]:
def aggregate_results(results: List[Dict[str, Any]], out_dir: Path):
    per_image_csv = out_dir / "per_image_summary.csv"
    with open(per_image_csv, "w", newline="", encoding="utf-8") as f:
        w = csv.writer(f)
        w.writerow([
            "image","winner","strength",
            "A_final","B_final",
            "A_accuracy","B_accuracy",
            "A_relation_correctness","B_relation_correctness",
            "A_completeness","B_completeness"
        ])
        for r in results:
            meta = r["final"]
            pair = meta["final_pairwise"]
            A = meta["final_pointwise"]["A"]["final_score"]
            B = meta["final_pointwise"]["B"]["final_score"]
            w.writerow([
                r["image"], pair["preferred_response"], pair["preference_strength"],
                A, B,
                meta["final_pointwise"]["A"]["accuracy"], meta["final_pointwise"]["B"]["accuracy"],
                meta["final_pointwise"]["A"]["relation_correctness"], meta["final_pointwise"]["B"]["relation_correctness"],
                meta["final_pointwise"]["A"]["completeness"], meta["final_pointwise"]["B"]["completeness"],
            ])

    wins = {"A":0,"B":0,"tie":0}
    a_scores, b_scores = [], []
    for r in results:
        pr = r["final"]["final_pairwise"]["preferred_response"]
        wins[pr] += 1
        a_scores.append(r["final"]["final_pointwise"]["A"]["final_score"])
        b_scores.append(r["final"]["final_pointwise"]["B"]["final_score"])

    report = {
        "count": len(results),
        "wins": wins,
        "avg_final_A": round(sum(a_scores)/max(1,len(a_scores)),3),
        "avg_final_B": round(sum(b_scores)/max(1,len(b_scores)),3)
    }
    with open(out_dir / "global_summary.json", "w", encoding="utf-8") as f:
        json.dump(report, f, indent=2)
    print(json.dumps(report, indent=2))


#### Cell 12 — Run the full pipeline (mock adapters by default)

In [ ]:
# Toggle mock for offline testing if you want
use_mock = False

if use_mock:
    gemma = MockAdapter()  # Judge-1
    qwen  = MockAdapter()  # Meta-Judge
else:
    # Judge-1: Gemma on AvalAI
    gemma = AvalAIAdapter(
        model_name=GEMMA_MODEL, temperature=0.0,
        base_url=AVALAI_BASE_URL, secret_name=AVALAI_SECRET_NAME
    )
    # Meta-Judge: Qwen on AvalAI
    qwen = AvalAIAdapter(
        model_name=QWEN_MODEL, temperature=0.0,
        base_url=AVALAI_BASE_URL, secret_name=AVALAI_SECRET_NAME
    )

results = []
for img_name in tqdm(overlap, desc="Evaluating overlap"):
    img_b64 = None
    if IMAGES_DIR.exists():
        ib = load_image_bytes(IMAGES_DIR, img_name)
        if ib:
            img_b64 = image_to_base64_jpeg(ib)
    try:
        out = run_pipeline_for_image(
            image_name=img_name,
            turtle_a=map_few[img_name],  # Few-shot as A
            turtle_b=map_one[img_name],  # One-shot as B
            image_b64_jpeg=img_b64,
            gemma=gemma,
            qwen=qwen,
            out_dir=OUT_DIR
        )
        results.append(out)
    except (ValidationError, ValueError, RuntimeError) as e:
        print(f"[ERROR] {img_name}: {e}")

aggregate_results(results, OUT_DIR)
print("Per-image JSONs in:", OUT_DIR)


### Cell 13 — Peek at outputs

In [ ]:
import pandas as pd

summary_csv = OUT_DIR / "per_image_summary.csv"
if summary_csv.exists():
    display(pd.read_csv(summary_csv).head(10))

global_json = OUT_DIR / "global_summary.json"
if global_json.exists():
    print(global_json.read_text()[:500])

samples = sorted(glob.glob(str(OUT_DIR / "*.final.json")))
print("Sample file:", samples[0] if samples else "None")
if samples:
    print(Path(samples[0]).read_text()[:800])


### Cell 14 — (Optional) Switch to real LLMs

In [ ]:
import time

def with_retries(adapter: AvalAIAdapter, prompt: str, image_b64_jpeg: Optional[str] = None,
                 tries: int = 3, delay: float = 1.5) -> str:
    last_err = None
    for i in range(tries):
        try:
            return adapter.generate(prompt, image_b64_jpeg=image_b64_jpeg, response_format_json=True)
        except Exception as e:
            last_err = e
            time.sleep(delay)
    raise last_err

# Example usage:
# text = with_retries(gemma, "hello")
# For full integration, you could swap adapter.generate(...) calls with with_retries(...)
